In [1]:
import cv2
import numpy as np
import datetime

class AdvancedObjectCounter:
    def __init__(self, video_source=0, output_file='annotated_output.avi'):
        """
        Initializes the video capture, background subtractor, and video writer.
        """
        # Capture video from webcam or video file [cite: 7]
        self.cap = cv2.VideoCapture(video_source)

        # Apply background subtraction to detect moving objects [cite: 7]
        # High-Tech tweak: history and varThreshold tuned for better accuracy
        self.bg_subtractor = cv2.createBackgroundSubtractorMOG2(history=500, varThreshold=50, detectShadows=True)

        # Setup video writer to save annotated video output [cite: 8]
        frame_width = int(self.cap.get(cv2.CAP_PROP_FRAME_WIDTH))
        frame_height = int(self.cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
        fps = int(self.cap.get(cv2.CAP_PROP_FPS)) or 30

        self.out = cv2.VideoWriter(output_file, cv2.VideoWriter_fourcc(*'XVID'), fps, (frame_width, frame_height))

        self.count = 0
        # Dynamic line positioning for counting
        self.line_y = frame_height // 2 + 50

    def process_stream(self):
        """
        Main loop to process frames, detect objects, and update the counter.
        """
        print("[INFO] Starting video stream... Press 'ESC' to exit.")

        while self.cap.isOpened():
            ret, frame = self.cap.read()
            if not ret:
                break

            # Get background mask
            fg_mask = self.bg_subtractor.apply(frame)

            # --- OUT OF THE BOX FEATURE: Morphological Cleaning ---
            # Removes random camera noise so bugs/dust aren't counted
            kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5))
            fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_OPEN, kernel)
            fg_mask = cv2.morphologyEx(fg_mask, cv2.MORPH_CLOSE, kernel)

            # Find contours of moving objects
            contours, _ = cv2.findContours(fg_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

            # Draw the counting reference line
            cv2.line(frame, (0, self.line_y), (frame.shape[1], self.line_y), (0, 255, 255), 2)

            for cnt in contours:
                # Filter out small movements
                if cv2.contourArea(cnt) > 3000:
                    x, y, w, h = cv2.boundingRect(cnt)

                    # Draw bounding boxes around detected objects [cite: 7]
                    cv2.rectangle(frame, (x, y), (x+w, y+h), (0, 255, 0), 2)

                    # Calculate centroid for tracking
                    cx, cy = x + w//2, y + h//2
                    cv2.circle(frame, (cx, cy), 5, (0, 0, 255), -1)

                    # Counting logic: If centroid touches the line
                    if self.line_y - 6 < cy < self.line_y + 6:
                        self.count += 1
                        # Flash the line red when an object crosses
                        cv2.line(frame, (0, self.line_y), (frame.shape[1], self.line_y), (0, 0, 255), 4)

            # Maintain and display a live object counter [cite: 8]
            # --- OUT OF THE BOX FEATURE: Professional HUD ---
            cv2.rectangle(frame, (10, 10), (350, 110), (0, 0, 0), -1) # Dark background for text
            cv2.putText(frame, f"Live Object Count: {self.count}", (20, 50),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)
            cv2.putText(frame, f"Status: Active | Time: {datetime.datetime.now().strftime('%H:%M:%S')}",
                        (20, 90), cv2.FONT_HERSHEY_SIMPLEX, 0.5, (0, 255, 255), 1)

            # Save frame to output file [cite: 8]
            self.out.write(frame)

            # Display the result
            cv2.imshow("Advanced Object Tracker & Counter", frame)

            # Exit condition
            if cv2.waitKey(30) & 0xFF == 27: # 27 is the ESC key
                break

        # Cleanup
        self.cap.release()
        self.out.release()
        cv2.destroyAllWindows()
        print(f"[INFO] Video saved successfully. Total objects counted: {self.count}")

if __name__ == "__main__":
    # To use a video file instead of webcam, pass the path: video_source='traffic.mp4'
    tracker = AdvancedObjectCounter(video_source=0)
    tracker.process_stream()

[INFO] Starting video stream... Press 'ESC' to exit.
[INFO] Video saved successfully. Total objects counted: 73
